# Module 03: BIDS Validation with bids-validator and PyBIDS

**Prerequisites:** Modules 00–02 | **Estimated time:** 1–2 hours

---

## 1. Why Validate?

The BIDS specification is only useful if your dataset actually conforms to it.
Validation matters because:

* **Analysis tools** (fMRIPrep, MRIQC, …) rely on a valid BIDS layout to find
  files automatically. A missing field map sidecar or a wrong filename means a
  critical preprocessing step is silently skipped.
* **Data sharing** platforms like OpenNeuro reject datasets that fail validation.
* **Reproducibility** — a dataset that passes the validator will be interpretable
  years later by anyone, not just the original analyst.

### Two complementary tools

| Tool | What it checks | When to use |
|------|---------------|-------------|
| **bids-validator** | Full spec compliance: filenames, JSON fields, required files | Before sharing; after every conversion run |
| **PyBIDS** | Programmatic queries; custom completeness checks | During analysis; automated QC scripts |

> **Note:** This notebook shows both tools. When working with your own data,
> set `BIDS_DIR` to the root of your BIDS dataset.

In [ ]:
import subprocess
import json
import os
import pathlib
import sys

# Point this at your BIDS dataset root
BIDS_DIR = pathlib.Path(os.environ.get("FMRI_TUTORIAL_BIDS_DIR", "/data/bids_output"))

print(f"BIDS_DIR: {BIDS_DIR}")
print(f"Exists  : {BIDS_DIR.exists()}")

# Check bids-validator availability
validator_check = subprocess.run(
    ["bids-validator", "--version"], capture_output=True, text=True
)
if validator_check.returncode == 0:
    print(f"bids-validator: {validator_check.stdout.strip()}")
else:
    print("bids-validator: NOT FOUND")
    print("  Install with: npm install -g bids-validator")

# Check PyBIDS availability
try:
    import bids
    print(f"PyBIDS        : {bids.__version__}")
except ImportError:
    print("PyBIDS        : NOT FOUND — pip install pybids")

---

## 2. Running bids-validator from the Command Line

The canonical way to run bids-validator is:

```bash
bids-validator /path/to/bids_dataset
```

Useful flags:

| Flag | Effect |
|------|--------|
| `--json` | Machine-readable JSON output |
| `--ignoreWarnings` | Suppress warnings (only show errors) |
| `--ignoreNiftiHeaders` | Skip NIfTI header checks (faster) |
| `--verbose` | More detail per issue |
| `--config` | Path to a custom `.bids-validator-config.json` |

We will run it programmatically below using `subprocess` so we can parse the
output in this notebook.

In [ ]:
def run_bids_validator(bids_dir: pathlib.Path, json_output: bool = False) -> dict | str:
    """Run bids-validator and return parsed JSON or raw text."""
    cmd = ["bids-validator", str(bids_dir)]
    if json_output:
        cmd.append("--json")

    result = subprocess.run(cmd, capture_output=True, text=True)
    combined = result.stdout + result.stderr

    if json_output:
        try:
            return json.loads(combined)
        except json.JSONDecodeError:
            return {"raw": combined, "returncode": result.returncode}
    return combined


if BIDS_DIR.exists():
    print(f"Running: bids-validator {BIDS_DIR}")
    output = run_bids_validator(BIDS_DIR)
    print(output[-5000:] if len(output) > 5000 else output)
else:
    print(f"[SKIP] BIDS_DIR not found: {BIDS_DIR}")

---

## 3. Understanding bids-validator Output

bids-validator distinguishes between **errors** (must fix before the dataset is
considered valid) and **warnings** (best-practice violations that may be
acceptable).

### Common errors and fixes

| Code | Message | Typical cause | Fix |
|------|---------|--------------|-----|
| `NOT_INCLUDED` | File not recognised | Extra file in dataset | Add to `.bidsignore` |
| `EVENTS_TSV_MISSING` | No events file for BOLD run | Task data without events | Add `_events.tsv` |
| `BOLD_NOT_4D` | 3-D NIfTI for BOLD series | Single-volume BOLD | Re-run dcm2niix |
| `MISSING_SESSION` | Mixed session / non-session layout | Partial session adoption | Use sessions for all or none |
| `INVALID_JSON` | Malformed sidecar | Encoding or syntax error | Check with `python -m json.tool` |

### JSON output format

With `--json`, the validator returns:
```json
{
  "issues": {
    "errors": [{"key": "NOT_INCLUDED", "severity": "error", "files": [...]}],
    "warnings": [{"key": "EVENTS_TSV_MISSING", "severity": "warning", ...}]
  },
  "summary": {"sessions": [], "subjects": ["sub-01"], "tasks": ["rest"], ...}
}
```

In [ ]:
def summarise_validator_json(report: dict) -> None:
    """Print a concise summary of a bids-validator JSON report."""
    if "raw" in report:
        print(report["raw"])
        return

    summary = report.get("summary", {})
    issues = report.get("issues", {})
    errors = issues.get("errors", [])
    warnings = issues.get("warnings", [])

    print("Dataset summary")
    print("=" * 40)
    print(f"  Subjects : {summary.get('subjects', [])}")
    print(f"  Sessions : {summary.get('sessions', 'n/a')}")
    print(f"  Tasks    : {summary.get('tasks', [])}")
    print(f"  Modalities: {summary.get('modalities', [])}")
    print()
    print(f"Errors   : {len(errors)}")
    for e in errors:
        print(f"  [{e.get('key','?')}] {e.get('reason','')}")
        for f in e.get("files", [])[:3]:
            print(f"    • {f.get('file',{}).get('relativePath','?')}")
    print()
    print(f"Warnings : {len(warnings)}")
    for w in warnings:
        print(f"  [{w.get('key','?')}] {w.get('reason','')}")


if BIDS_DIR.exists():
    print("Running bids-validator with --json flag ...")
    report = run_bids_validator(BIDS_DIR, json_output=True)
    summarise_validator_json(report)
else:
    print("[SKIP] BIDS_DIR not found — showing example report structure.")
    example = {
        "summary": {"subjects": ["sub-01"], "tasks": ["rest"], "modalities": ["func", "anat"]},
        "issues": {
            "errors": [],
            "warnings": [{"key": "EVENTS_TSV_MISSING",
                          "reason": "Events file not found for task rest.",
                          "files": [{"file": {"relativePath": "/sub-01/func/sub-01_task-rest_bold.nii.gz"}}]}],
        },
    }
    summarise_validator_json(example)

---

## 4. Using PyBIDS for Programmatic Validation

[PyBIDS](https://bids-standard.github.io/pybids/) provides a Python API to
query BIDS datasets. Its central object is `BIDSLayout`, which indexes the
entire dataset on first load.

```python
from bids import BIDSLayout

layout = BIDSLayout("/path/to/bids")
```

### `BIDSLayout.get()` filter arguments

| Argument | Example | Effect |
|----------|---------|--------|
| `subject` | `"01"` | Only subject `sub-01` |
| `task` | `"rest"` | Only task `rest` |
| `run` | `1` | Only run `01` |
| `suffix` | `"bold"` | Only BOLD files |
| `extension` | `".nii.gz"` | Only NIfTI |
| `datatype` | `"func"` | Only files in `func/` |
| `return_type` | `"filename"` | Return paths instead of objects |

In [ ]:
try:
    from bids import BIDSLayout
    PYBIDS_AVAILABLE = True
except ImportError:
    PYBIDS_AVAILABLE = False
    print("PyBIDS not installed. Run: pip install pybids")

layout = None

if PYBIDS_AVAILABLE and BIDS_DIR.exists():
    print(f"Loading BIDSLayout from {BIDS_DIR} ...")
    layout = BIDSLayout(str(BIDS_DIR), validate=False)
    print("Layout loaded.")
    print(f"  Subjects : {layout.get_subjects()}")
    print(f"  Sessions : {layout.get_sessions() or 'n/a'}")
    print(f"  Tasks    : {layout.get_tasks()}")
    print(f"  Runs     : {layout.get_runs() or 'n/a'}")
elif PYBIDS_AVAILABLE:
    print(f"[SKIP] BIDS_DIR not found: {BIDS_DIR}")
    print("Set FMRI_TUTORIAL_BIDS_DIR and re-run.")

---

## 5. Querying BOLD Files

Once the layout is loaded, use `layout.get()` to retrieve specific file types.

In [ ]:
if layout is not None:
    # All BOLD NIfTI files
    bold_files = layout.get(suffix="bold", extension=".nii.gz", return_type="filename")
    print(f"All BOLD files ({len(bold_files)}):")
    for f in bold_files:
        print(f"  {pathlib.Path(f).relative_to(BIDS_DIR)}")

    print()

    # Filter by subject
    subjects = layout.get_subjects()
    if subjects:
        sub = subjects[0]
        bold_sub = layout.get(
            subject=sub, suffix="bold", extension=".nii.gz", return_type="filename"
        )
        print(f"BOLD files for sub-{sub} ({len(bold_sub)}):")
        for f in bold_sub:
            print(f"  {pathlib.Path(f).relative_to(BIDS_DIR)}")

    print()

    # T1w anatomicals
    t1w_files = layout.get(suffix="T1w", extension=".nii.gz", return_type="filename")
    print(f"T1w files ({len(t1w_files)}):")
    for f in t1w_files:
        print(f"  {pathlib.Path(f).relative_to(BIDS_DIR)}")
else:
    print("[SKIP] BIDSLayout not available.")

---

## 6. Checking Dataset Completeness

A common QC task is to verify that every subject has the same set of files.
The cell below checks that each subject has at least one T1w, one BOLD run
per task, and a corresponding events file.

In [ ]:
def check_dataset_completeness(layout) -> None:
    """Report missing files across subjects."""
    subjects = layout.get_subjects()
    tasks = layout.get_tasks()
    issues = []

    print("Completeness check")
    print("=" * 60)
    print(f"  Subjects : {len(subjects)}")
    print(f"  Tasks    : {tasks or 'none'}")
    print()

    for subject in subjects:
        # Check T1w
        t1w = layout.get(subject=subject, suffix="T1w", extension=".nii.gz")
        if not t1w:
            issues.append(f"sub-{subject}: missing T1w")

        # Check BOLD and events per task
        for task in tasks:
            bold = layout.get(
                subject=subject, task=task, suffix="bold", extension=".nii.gz"
            )
            if not bold:
                issues.append(f"sub-{subject}: missing BOLD for task-{task}")
            else:
                # Check events file for each BOLD run
                for bf in bold:
                    entities = layout.get_file(bf.path).entities if hasattr(bf, 'path') else {}
                    run = entities.get("run", None)
                    events = layout.get(
                        subject=subject, task=task, run=run, suffix="events",
                        extension=".tsv",
                    )
                    if not events:
                        run_label = f" run-{run}" if run else ""
                        issues.append(
                            f"sub-{subject}: missing events.tsv for task-{task}{run_label}"
                        )

    if issues:
        print(f"Found {len(issues)} issue(s):")
        for issue in issues:
            print(f"  ✗ {issue}")
    else:
        print("All subjects complete ✓")


if layout is not None:
    check_dataset_completeness(layout)
else:
    print("[SKIP] BIDSLayout not available.")

---

## Summary

In this notebook you:

1. Checked that `bids-validator` and PyBIDS are installed.
2. Ran `bids-validator` programmatically and interpreted its error/warning output.
3. Parsed the JSON report to summarise issues concisely.
4. Loaded a `BIDSLayout` with PyBIDS and queried BOLD, T1w, and events files.
5. Wrote a completeness check that detects missing files across subjects.

### Next steps

* Use the `scripts/validate_dataset.sh` helper to integrate validation into your
  pipeline.
* Use `scripts/query_bids_pybids.py` to generate an automated dataset summary
  report.
* Proceed to **Module 04** for fMRIPrep preprocessing.

### Exercises

1. Open the JSON report file and identify which subjects have the most
   validation warnings.
2. Create a `.bidsignore` file to suppress `NOT_INCLUDED` warnings for your
   HeudiConv working files.
3. Use `layout.get_metadata()` to retrieve the `RepetitionTime` for all BOLD
   runs and verify it is consistent across subjects.